# Hidden Markov Model and Bayesian Network


## Introduction

Bayesian Networks (BNs) and Hidden Markov Models (HMMs) are both probabilistic graphical models used to represent complex probability distributions.
Bayesian Networks represent conditional dependencies between random variables in the form of a directed acyclic graph (DAG).
Hidden Markov Models, on the other hand, are a special type of Bayesian Network used for modeling sequential data with hidden states.

In this notebook, we'll explore:
1. The structure of Bayesian Networks and Hidden Markov Models.
2. The mathematical foundations of each model.
3. The key differences and similarities between Bayesian Networks and Hidden Markov Models.

We’ll also implement basic examples of each to illustrate their differences in structure and application.



## Bayesian Network

A Bayesian Network is a directed acyclic graph where each node represents a random variable, and each edge represents a conditional dependency.
Given a node $ X $ with parent nodes ${Parents}(X)$, the probability distribution of $ X $ is conditioned on its parents.

The joint probability distribution of a Bayesian Network with nodes $ X_1, X_2, ..., X_n $ is:
$$ P(X_1, X_2, ..., X_n) = \prod_{i=1}^n P(X_i | {Parents}(X_i)) $$

### Example
Consider a simple Bayesian Network where we have three random variables: `Rain`, `Sprinkler`, and `GrassWet`.

- `Rain` influences whether the `Sprinkler` is on.
- Both `Rain` and `Sprinkler` influence whether the `GrassWet` is true.

Below, we define this structure and use `pgmpy` for constructing the Bayesian Network.


In [ ]:
!pip install pgmpy
!pip install hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 5.5 MB/s eta 0:00:00


In [ ]:

from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Define the structure of the Bayesian Network
model = BayesianNetwork([('Rain', 'Sprinkler'), ('Rain', 'GrassWet'), ('Sprinkler', 'GrassWet')])

# Define the conditional probability distributions (CPDs)
cpd_rain = TabularCPD(variable='Rain', variable_card=2, values=[[0.7], [0.3]])
cpd_sprinkler = TabularCPD(variable='Sprinkler', variable_card=2,
                           values=[[0.8, 0.1], [0.2, 0.9]],
                           evidence=['Rain'], evidence_card=[2])
cpd_grasswet = TabularCPD(variable='GrassWet', variable_card=2,
                          values=[[0.99, 0.9, 0.8, 0.0], [0.01, 0.1, 0.2, 1.0]],
                          evidence=['Sprinkler', 'Rain'], evidence_card=[2, 2])

# Add CPDs to the model
model.add_cpds(cpd_rain, cpd_sprinkler, cpd_grasswet)

# Perform inference
inference = VariableElimination(model)
prob_grasswet = inference.query(variables=['GrassWet'], evidence={'Rain': 1, 'Sprinkler': 0})
print(prob_grasswet)


+-------------+-----------------+
| GrassWet    |   phi(GrassWet) |
+=============+=================+
| GrassWet(0) |          0.9000 |
+-------------+-----------------+
| GrassWet(1) |          0.1000 |
+-------------+-----------------+



## Hidden Markov Model

An HMM is a statistical model where the system being modeled is assumed to be a Markov process with hidden states.
It consists of:
1. **Hidden States** - The actual (hidden) states that the process moves through.
2. **Observations** - Observed values that depend on the hidden states.

The HMM consists of three main parameters:
- Transition probabilities between states.
- Emission probabilities for each state.
- Initial probabilities for each state.

For a sequence of observations $ O = O_1, O_2, ..., O_T $ and hidden states $ S = S_1, S_2, ..., S_T $, the joint probability can be written as:
$$ P(O, S) = P(S_1) \prod_{t=2}^T P(S_t | S_{t-1}) \prod_{t=1}^T P(O_t | S_t) $$

### Example
Below is an implementation of a simple HMM with two hidden states and observable states.


In [ ]:

import numpy as np
from hmmlearn import hmm

# Define model parameters
n_states = 2
n_observations = 3

# Transition and emission matrices
transition_probs = np.array([[0.7, 0.3], [0.4, 0.6]])
emission_probs = np.array([[0.1, 0.4, 0.5], [0.6, 0.3, 0.1]])
start_probs = np.array([0.6, 0.4])

# Build HMM model
model = hmm.MultinomialHMM(n_components=n_states)
model.startprob_ = start_probs
model.transmat_ = transition_probs
model.emissionprob_ = emission_probs

# Set the n_trials parameter (e.g., to 1 for a single trial per observation)
# This assumes that each observation is a single outcome (like a coin flip)
model.n_trials = 1


# Generate a sequence of observations
observed_sequence, hidden_states = model.sample(10)

print("Observed sequence:", observed_sequence.flatten())
print("Hidden states:", hidden_states.flatten())


https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


Observed sequence: [0 1 0 0 1 0 0 1 0 0 1 0 0 0 1 0 1 0 0 1 0 0 0 1 0 0 1 0 0 1]
Hidden states: [0 0 0 1 0 0 0 0 0 0]


## Relationship Between Hidden Markov Models and Bayesian Networks

Hidden Markov Models (HMMs) and Bayesian Networks (BNs) are both probabilistic graphical models, but they are structured differently to address distinct types of data and dependencies.

### HMM as a Special Case of Bayesian Network

An HMM can be viewed as a Bayesian Network with a sequential or chain structure, where each hidden state is conditionally dependent on the previous hidden state. This structure follows the **Markov property**, which states that the current state is independent of all previous states given the immediately preceding state. This simplifies the model by focusing only on **direct dependencies** between consecutive hidden states rather than all past states.

In other words, an HMM is a type of **Dynamic Bayesian Network (DBN)**. DBNs are Bayesian Networks that evolve over time, making them ideal for modeling sequences or time-series data. HMMs are one of the simplest forms of DBNs, designed specifically to represent a sequence of **hidden states** and **observations** over time.

### Structure of HMM

In an HMM, the process can be visualized as a series of time steps $t = 1, 2, ..., T$, with two types of variables:
1. **Hidden States** ($S_t$) - These are the actual states of the system at each time step $t$, which are not directly observable.
2. **Observations** ($O_t$) - These are the observations made at each time step, which depend on the hidden state at that time.

Each hidden state $S_t$ is conditionally dependent on:
- The **previous hidden state** $S_{t-1}$ (transition probabilities).
- The **current observation** $O_t$ (emission probabilities).

The **Markov property** allows us to express the joint probability of a sequence of states and observations efficiently, using only the dependencies between consecutive time steps.

### Joint Probability Representation

For a sequence of observations $O = O_1, O_2, ..., O_T$ and hidden states $S = S_1, S_2, ..., S_T$, the joint probability of the entire sequence can be expressed as:
$$
P(O, S) = P(S_1) \prod_{t=2}^T P(S_t | S_{t-1}) \prod_{t=1}^T P(O_t | S_t)
$$

This joint probability is broken down into:
- $P(S_1)$: The initial probability distribution over the hidden states.
- $P(S_t | S_{t-1})$: Transition probabilities between consecutive hidden states.
- $P(O_t | S_t)$: Emission probabilities for the observations given the hidden states.

### Key Differences Between Bayesian Networks and HMMs

1. **General Structure vs. Sequential Structure**: Bayesian Networks can represent any conditional dependency structure, whereas HMMs are specifically structured to model sequences with temporal dependencies.

2. **Static vs. Dynamic**: Standard Bayesian Networks are static, meaning they do not change over time. HMMs, as a form of Dynamic Bayesian Network, are designed to model evolving sequences, making them ideal for applications like speech recognition, natural language processing, and time-series analysis.

3. **Hidden States and Observations**: HMMs explicitly differentiate between hidden states and observations. Bayesian Networks do not necessarily have this structure and can represent any conditional dependency relationships.

### Summary

In summary, an HMM is a specialized Bayesian Network optimized for modeling sequential data with hidden states that evolve over time. This makes HMMs a powerful tool for applications where the underlying states are not directly observable but can be inferred through observable data.



## Relationship between Bayesian Networks and Hidden Markov Models

An HMM can be viewed as a Bayesian Network with a chain structure, where each hidden state is conditionally dependent on the previous state (Markov property).
In this sense, an HMM is a dynamic Bayesian Network that represents a sequence of hidden states and observations over time.

Key differences:
- **Bayesian Network**: General purpose, models any kind of dependencies.
- **Hidden Markov Model**: Specialized, models sequential dependencies with hidden states.

### Summary
Both models use conditional dependencies to model probabilities, but HMMs are specifically structured for time series with hidden states.
